# Seq2seq mapper: transcript ASR → reference (`audio_text`)

Fine-tune **ViT5 / BARTpho** để map câu ASR (chữ thường, không dấu câu, số đọc bằng chữ) ra đúng câu reference:
sửa lỗi nhận dạng + ITN (`năm mươi nghìn` → `50 nghìn`) + viết hoa/dấu câu trong một bước.

| | |
|---|---|
| Dữ liệu | `train.jsonl`: query + mọi lượt user trong `history` (~5036 câu) — cặp (hyp Zipformer, `audio_text`) |
| Target | `spoken` (mặc định): `ref_spoken` — dạng đọc tạo bằng `mapper/tn_agy.py` (agy, ưu tiên cách đọc số của người nói); mapper chỉ sửa lỗi ASR, **ITN của pipeline** chuyển sang dạng viết sau |
| Hyp | đọc từ `asr_train_tn.jsonl` / `asr_train.jsonl` (tạo bằng `python mapper/seq2seq_mapper.py hyps ... --bias` ở local) **hoặc** chạy Zipformer greedy ngay trong notebook |
| Chia | theo **hội thoại** (dev 10%) |
| Augment | thêm cặp `norm(ref) → ref`: học ITN/dấu câu và học "câu đúng thì giữ nguyên" |
| Output | thư mục HF model + `mapper_config.json`, dùng được với `Seq2SeqMapper` trong `mapper/seq2seq_mapper.py` |

⚠️ Hyp nên lấy từ Zipformer **gốc** (hoặc fine-tune theo k-fold). Zipformer đã fine-tune trên chính train cho hyp quá sạch → mapper học được ít.

## 1. Cấu hình

In [ ]:
from pathlib import Path

DATA_DIR   = Path("/content/train")          # chứa train.jsonl, audio/ (Kaggle: /kaggle/input/<dataset>/train)
WORK       = Path("/content/s2s_mapper")     # checkpoint + model cuối (Kaggle: /kaggle/working/s2s_mapper)
ASR_JSONL  = Path("/content/asr_train_tn.jsonl")  # cặp (hyp, ref[, ref_spoken]); TARGET="spoken" cần file từ mapper/tn_agy.py
PUBLIC_ASR = Path("/content/asr_public.jsonl") # (tuỳ chọn) output asr/test_asr_public.py để predict thử
ZIPFORMER  = "hynt/Zipformer-30M-RNNT-6000h"

MODEL_NAME = "VietAI/vit5-base"   # hoặc "vinai/bartpho-syllable" (khi đó LR = 5e-5)
TARGET     = "spoken"             # "spoken" = ref_spoken (dạng đọc, ITN làm sau — khuyến nghị); "raw" = đúng audio_text; "norm"
EPOCHS, LR, BS, MAX_LEN = 10, 3e-4, 16, 128
DEV_FRAC, SEED, BEAMS   = 0.1, 0, 4
AUG_CLEAN = True
MODEL_OUT = WORK / "mapper_s2s"
WORK.mkdir(parents=True, exist_ok=True)

**(Colab)** đưa dữ liệu lên: nén thư mục `train` thành zip, để trên Google Drive rồi giải nén:

In [ ]:
# from google.colab import drive; drive.mount("/content/drive")
# !unzip -q "/content/drive/MyDrive/train.zip" -d /content/
# !cp "/content/drive/MyDrive/asr_train_tn.jsonl" /content/   # tạo ở local: mapper/seq2seq_mapper.py hyps -> mapper/tn_agy.py
assert (DATA_DIR / "train.jsonl").exists() or ASR_JSONL.exists(), "Cần train.jsonl hoặc asr_train.jsonl"

## 2. Cài đặt

In [ ]:
import sys, subprocess, torch
def pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])
pip("transformers>=4.45", "datasets", "accelerate", "sentencepiece", "protobuf", "sherpa-onnx", "soundfile", "huggingface_hub")
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

## 3. Tiện ích: chuẩn hoá + WER/CER

In [ ]:
import json, random, re

def norm(s):
    """Dạng giống output ASR: chữ thường, bỏ dấu câu (giữ '.'/',' giữa hai chữ số: 2.5)."""
    s = re.sub(r"(?<!\d)[^\w\s]|[^\w\s](?!\d)", " ", s.lower())
    return " ".join(s.split())

def edit_counts(ref, hyp):
    r, h = ref.split(), hyp.split()
    d = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(h) + 1):
            cur = min(d[j] + 1, d[j - 1] + 1, prev + (r[i - 1] != h[j - 1]))
            prev, d[j] = d[j], cur
    return d[len(h)], len(r)

def score(pairs):
    """pairs: [(ref, pred)]. WER/CER tính sau norm, exact match trên dạng gốc."""
    we = wn = ce = cn = em = 0
    chars = lambda s: " ".join(norm(s).replace(" ", "_"))
    for ref, pred in pairs:
        e, n = edit_counts(norm(ref), norm(pred)); we += e; wn += n
        e, n = edit_counts(chars(ref), chars(pred)); ce += e; cn += n
        em += ref.strip() == pred.strip()
    return {"wer": we / max(wn, 1), "cer": ce / max(cn, 1), "exact": em / max(len(pairs), 1), "n": len(pairs)}

assert norm("Giá 2.5 triệu, nhé!") == "giá 2.5 triệu nhé"
assert edit_counts("a b c", "a x c") == (1, 3)

## 4. Cặp (hyp, ref)
Có `asr_train.jsonl` thì dùng luôn (khuyến nghị: tạo ở local với `--bias` để hyp giống hệt input của pipeline).
Không có thì chạy Zipformer **greedy** (sherpa-onnx, CPU) trên mọi lượt user — ~5k câu, vài chục phút.

In [ ]:
if not ASR_JSONL.exists():
    import os, soundfile as sf, sherpa_onnx, sentencepiece as spm
    from huggingface_hub import snapshot_download
    zd = Path(snapshot_download(ZIPFORMER, allow_patterns=["*.onnx", "bpe.model", "tokens.txt"]))
    tok_file = zd / "tokens.txt"
    if not tok_file.exists():  # repo chỉ có bpe.model -> tạo tokens.txt
        sp = spm.SentencePieceProcessor(model_file=str(zd / "bpe.model"))
        tok_file = WORK / "tokens.txt"
        tok_file.write_text("".join(f"{sp.id_to_piece(i)} {i}\n" for i in range(sp.get_piece_size())), encoding="utf-8")
    pick = lambda n: str(next(p for p in sorted(zd.glob(f"{n}-*.onnx")) if not p.name.endswith(".int8.onnx")))
    rec = sherpa_onnx.OfflineRecognizer.from_transducer(
        encoder=pick("encoder"), decoder=pick("decoder"), joiner=pick("joiner"), tokens=str(tok_file),
        num_threads=os.cpu_count(), sample_rate=16000, feature_dim=80, decoding_method="greedy_search")

    items = [json.loads(l) for l in open(DATA_DIR / "train.jsonl", encoding="utf-8") if l.strip()]
    with open(ASR_JSONL, "w", encoding="utf-8") as f:
        for si, it in enumerate(items, 1):
            turns = [h for h in it["history"] if h["role"] == "user"] + [it]
            streams = []
            for u in turns:
                x, sr = sf.read(str(DATA_DIR / u["audio"]), dtype="float32")
                s = rec.create_stream(); s.accept_waveform(sr, x.mean(1) if x.ndim > 1 else x); streams.append(s)
            rec.decode_streams(streams)
            for k, (u, s) in enumerate(zip(turns, streams)):
                f.write(json.dumps({"id": it["id"], "turn": "QUERY" if k == len(turns) - 1 else f"u{k + 1}",
                                    "audio": u["audio"], "hyp": s.result.text.strip().lower(),
                                    "ref": u.get("audio_text")}, ensure_ascii=False) + "\n")
            if si % 100 == 0: print(f"{si}/{len(items)}")
    print("->", ASR_JSONL)

In [ ]:
rows = [json.loads(l) for l in open(ASR_JSONL, encoding="utf-8") if l.strip()]
rows = [r for r in rows if r.get("ref") and r.get("hyp") is not None]
if TARGET == "spoken":
    assert all("ref_spoken" in r for r in rows), "thiếu ref_spoken: chạy mapper/tn_agy.py ở local trước"
    rows = [r for r in rows if r.get("tn_ok", True)]
for r in rows:
    r["src"] = norm(r["hyp"])
    r["tgt"] = {"norm": lambda: norm(r["ref"]), "spoken": lambda: r["ref_spoken"],
                "raw": lambda: " ".join(r["ref"].split())}[TARGET]()

ids = sorted({r["id"] for r in rows}); random.Random(SEED).shuffle(ids)
dev_ids = set(ids[:max(1, int(len(ids) * DEV_FRAC))])
train = [r for r in rows if r["id"] not in dev_ids]
dev   = [r for r in rows if r["id"] in dev_ids]
clean = (lambda r: r["tgt"]) if TARGET == "spoken" else (lambda r: norm(r["ref"]))   # spoken: cặp đồng nhất
aug = [{"src": clean(r), "tgt": r["tgt"]} for r in train] if AUG_CLEAN else []
train_pairs = [{"src": r["src"], "tgt": r["tgt"]} for r in train] + aug
print(f"train {len(train)} (+{len(aug)} aug) | dev {len(dev)} | target={TARGET}")

base = score([(r["tgt"], r["src"]) for r in dev])
print(f"baseline (giữ nguyên hyp): WER {base['wer']:.4f}  CER {base['cer']:.4f}  exact {base['exact']:.3f}")
for r in dev[:3]: print(" hyp:", r["src"], "\n ref:", r["tgt"])

## 5. Model + tokenize

In [ ]:
import numpy as np
from datasets import Dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq, PreTrainedTokenizerFast

try:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
except TypeError:
    # transformers 5: T5Tokenizer ép Unigram, nhưng tokenizer.json của ViT5 là BPE -> nạp thẳng tokenizer.json
    tok = PreTrainedTokenizerFast.from_pretrained(MODEL_NAME)
print(type(tok).__name__, "| eos", tok.eos_token, tok.eos_token_id, "| pad", tok.pad_token, tok.pad_token_id)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def enc(b):
    x = tok(b["src"], max_length=MAX_LEN, truncation=True)
    x["labels"] = tok(text_target=b["tgt"], max_length=MAX_LEN, truncation=True)["input_ids"]
    return x

tr = Dataset.from_list(train_pairs).map(enc, batched=True, remove_columns=["src", "tgt"])
dv = Dataset.from_list([{"src": r["src"], "tgt": r["tgt"]} for r in dev]).map(enc, batched=True, remove_columns=["src", "tgt"])
lens = [len(x) for x in tr["labels"]]
print(f"label tokens: max {max(lens)}, p99 {int(np.percentile(lens, 99))} (MAX_LEN={MAX_LEN})")
# câu khó nhất phải tokenize->decode lại được (ViT5 vocab có đủ dấu tiếng Việt)
print(tok.decode(tok(train_pairs[0]["tgt"])["input_ids"], skip_special_tokens=True))

## 6. Train
T5 + fp16 dễ NaN → T5 chỉ dùng bf16 (A100/L4) hoặc fp32 (T4). BARTpho thì fp16 được.

In [ ]:
import inspect
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16 = torch.cuda.is_available() and not bf16 and "t5" not in model.config.model_type
warm = ({"warmup_ratio": 0.05} if "warmup_ratio" in inspect.signature(Seq2SeqTrainingArguments).parameters
        else {"warmup_steps": 0.05})   # transformers 5: warmup_steps float = tỉ lệ

def metrics(p):
    preds, labels = p
    preds = np.where(preds < 0, tok.pad_token_id, preds)
    labels = np.where(labels < 0, tok.pad_token_id, labels)
    return score(list(zip(tok.batch_decode(labels, skip_special_tokens=True),
                          tok.batch_decode(preds, skip_special_tokens=True))))

args = Seq2SeqTrainingArguments(
    output_dir=str(WORK / "ckpt"), num_train_epochs=EPOCHS, learning_rate=LR, weight_decay=0.01,
    per_device_train_batch_size=BS, per_device_eval_batch_size=BS * 2, **warm,
    lr_scheduler_type="linear", logging_steps=50, eval_strategy="epoch", save_strategy="epoch",
    save_total_limit=1, load_best_model_at_end=True, metric_for_best_model="wer", greater_is_better=False,
    predict_with_generate=True, generation_max_length=MAX_LEN, generation_num_beams=1,
    bf16=bf16, fp16=fp16, seed=SEED, report_to="none")
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=tr, eval_dataset=dv, processing_class=tok,
                         data_collator=DataCollatorForSeq2Seq(tok, model=model), compute_metrics=metrics)
trainer.train()

trainer.save_model(str(MODEL_OUT)); tok.save_pretrained(str(MODEL_OUT))
json.dump({"target": TARGET, "max_len": MAX_LEN, "base": MODEL_NAME}, open(MODEL_OUT / "mapper_config.json", "w"))
print("saved ->", MODEL_OUT)

## 7. Đánh giá dev (beam search + guard)
Guard: output dài/ngắn bất thường so với input (ảo giác, cụt câu) → trả lại input.

In [ ]:
model = trainer.model.eval()
dev_device = model.device

@torch.no_grad()
def map_texts(texts, beams=BEAMS, bs=32, min_ratio=0.6, max_ratio=1.6):
    out = []
    for i in range(0, len(texts), bs):
        chunk = [norm(t) for t in texts[i:i + bs]]
        x = tok(chunk, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(dev_device)
        gen = model.generate(**x, max_new_tokens=MAX_LEN, num_beams=beams, do_sample=False)
        for src, pred in zip(chunk, tok.batch_decode(gen, skip_special_tokens=True)):
            ns, npred = len(src.split()), len(norm(pred).split())
            out.append(pred.strip() if ns == 0 or min_ratio <= npred / ns <= max_ratio else src)
    return out

preds = map_texts([r["src"] for r in dev])
s = score([(r["tgt"], p) for r, p in zip(dev, preds)])
print(f"baseline WER {base['wer']:.4f} CER {base['cer']:.4f} | mapper WER {s['wer']:.4f} CER {s['cer']:.4f} exact {s['exact']:.3f}")

worse = [(r, p) for r, p in zip(dev, preds) if edit_counts(norm(r["tgt"]), norm(p))[0] > edit_counts(norm(r["tgt"]), r["src"])[0]]
better = [(r, p) for r, p in zip(dev, preds) if edit_counts(norm(r["tgt"]), norm(p))[0] < edit_counts(norm(r["tgt"]), r["src"])[0]]
print(f"tốt hơn hyp: {len(better)} câu | tệ hơn hyp: {len(worse)} câu / {len(dev)}")
for tag, lst in (("TỆ HƠN", worse), ("TỐT HƠN", better)):
    print(f"\n===== {tag}")
    for r, p in lst[:8]: print(f"  hyp : {r['src']}\n  ref : {r['tgt']}\n  pred: {p}\n")

## 8. Predict thử
Câu tự gõ, và (nếu có) `asr_public.jsonl` từ `asr/test_asr_public.py` — trường `bias`.

In [ ]:
for t in ["đặt báo thức năm giờ rưỡi sáng mai nhé",
          "nạp giúp mình năm mươi ngàn tiền điện thoại cho bác ba",
          "lịch sang tiếng anh giúp tôi câu này"]:
    print(t, "\n ->", map_texts([t])[0])

if PUBLIC_ASR.exists():
    pub = [json.loads(l) for l in open(PUBLIC_ASR, encoding="utf-8") if l.strip()]
    for r, p in zip(pub, map_texts([r["bias"] or "" for r in pub])): r["mapped"] = p
    with open(WORK / "public_mapped.jsonl", "w", encoding="utf-8") as f:
        for r in pub: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    for r in pub[:10]: print(f"\n  asr : {r['bias']}\n  map : {r['mapped']}")

## 9. Đóng gói
Tải zip về, giải nén rồi dùng: `Seq2SeqMapper("mapper_s2s")` hoặc `python mapper/seq2seq_mapper.py predict --model mapper_s2s ...`

In [ ]:
import shutil
z = shutil.make_archive(str(WORK / "mapper_s2s"), "zip", MODEL_OUT)
print(z)
# from google.colab import files; files.download(z)